In [2]:
"""
Optimized MSI Spatial Visualization Script - Fixed Version

Features:
- Exact 60 µm pixel sizing with tight pixel arrangement
- Configurable vmax percentile parameter
- Works in both Jupyter notebooks and command line
- Fixed layout warnings and type hints
"""

import os
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from typing import Optional, Tuple, Dict, List, Any

# =============================================================================
# CONFIGURATION - MODIFY THESE PARAMETERS AS NEEDED
# =============================================================================

INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
COMMON_MZS_CSV_PATH = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/common_mz_clusters_improved.csv"
RESULTS_OUTPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/results/filtered_new_half/vmax/"

PIXEL_SIZE_UM = 60          # Pixel size in micrometers
VMAX_PERCENTILE = 99.9      # Percentile for vmax clipping (adjustable)
DPI = 100                   # Output image DPI

SAMPLE_ORDER = [
    'yc_1', 'yc_2', 'yc_3', 'yc_4',
    'yad_1', 'yad_2', 'yad_3', 'yad_4',
    'ac_1', 'ac_2', 'ac_3', 'ac_4',
    'aad_1', 'aad_2', 'aad_3', 'aad_4'
]

CUSTOM_CMAP = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),
    (0.00000001, "#000000"),
    (0.10, "#000080"),
    (0.15, "#0000FF"),
    (0.30, "#8000FF"),
    (0.45, "#FF0000"),
    (0.60, "#FF8000"),
    (0.75, "#FFFF00"),
    (1.0, "#FFFFFF")
])


# =============================================================================
# DATA LOADING
# =============================================================================

def load_samples(input_folder: str, sample_ids: List[str]) -> Dict[str, sc.AnnData]:
    """Load all h5ad sample files into a dictionary."""
    sample_map = {}
    for sample_id in sample_ids:
        filepath = os.path.join(input_folder, f"halfbrain_{sample_id}_filtered_common.h5ad")
        if os.path.exists(filepath):
            sample_map[sample_id] = sc.read_h5ad(filepath)
            print(f"Loaded: {sample_id}")
        else:
            print(f"Warning: File not found for {sample_id}")
    return sample_map


def compute_tic(sample_map: Dict[str, sc.AnnData]) -> None:
    """Compute and store TIC (Total Ion Current) for each sample."""
    for sample_id, adata in sample_map.items():
        tic = adata.X.sum(axis=1)
        if hasattr(tic, "A1"):
            tic = tic.A1
        else:
            tic = np.asarray(tic).flatten()
        adata.obs["Processed_TIC"] = tic


def load_and_prepare_pivot_table(csv_path: str, sample_order: List[str]) -> pd.DataFrame:
    """Load common m/z data and create sorted pivot table."""
    common_mz_df = pd.read_csv(csv_path)
    common_mz_df['sample_id'] = common_mz_df.apply(
        lambda row: f"{row['group'].lower()}_{row['sample']}", axis=1
    )
    pivot_df = common_mz_df.pivot(index='sample_id', columns='common_group_name', values='mzs')
    pivot_df_sorted = pivot_df.reindex(sample_order)
    return pivot_df_sorted


# =============================================================================
# DATA EXTRACTION
# =============================================================================

def extract_spatial_mz_data(
    adata: sc.AnnData,
    mz_val: float,
    tol: float = 0.01,
    normalize_tic: bool = False
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
    """Extract spatial coordinates and intensity data for a given m/z value."""
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - mz_val)
    
    if np.min(mz_diff) > tol:
        return None, None, None
    
    mz_index = np.argmin(mz_diff)
    
    if hasattr(adata.X, "toarray"):
        intensities = adata.X[:, mz_index].toarray().flatten()
    else:
        intensities = np.asarray(adata.X[:, mz_index]).flatten()
    
    x = adata.obs["x"].values.copy()
    y = adata.obs["y"].values.copy()
    
    if normalize_tic:
        tic = adata.obs["Processed_TIC"].values
        intensities = intensities / tic
        intensities = np.nan_to_num(intensities, nan=0.0, posinf=0.0, neginf=0.0)
    
    return x, y, intensities


# =============================================================================
# PLOTTING FUNCTIONS
# =============================================================================

def get_data_spacing(x: np.ndarray, y: np.ndarray, default: float = 60.0) -> float:
    """Calculate the grid spacing from coordinate data."""
    x_unique = np.sort(np.unique(x))
    y_unique = np.sort(np.unique(y))
    
    x_spacing = np.median(np.diff(x_unique)) if len(x_unique) > 1 else default
    y_spacing = np.median(np.diff(y_unique)) if len(y_unique) > 1 else default
    
    return min(x_spacing, y_spacing)


def calculate_marker_size(ax: plt.Axes, spacing: float, fig: plt.Figure) -> float:
    """Calculate marker size in points^2 for tight square pixels."""
    renderer = fig.canvas.get_renderer()
    bbox = ax.get_window_extent(renderer=renderer)
    ax_width_display = bbox.width
    ax_height_display = bbox.height
    
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    data_width = xlim[1] - xlim[0]
    data_height = ylim[1] - ylim[0]
    
    if data_width == 0 or data_height == 0:
        return 1.0
    
    display_per_data_x = ax_width_display / data_width
    display_per_data_y = ax_height_display / data_height
    display_per_data = min(display_per_data_x, display_per_data_y)
    
    dpi = fig.dpi
    pixel_points_size = spacing * display_per_data * 72 / dpi
    
    return pixel_points_size ** 2


def create_figure_with_gridspec(
    pivot_df: pd.DataFrame,
    sample_map: Dict[str, sc.AnnData],
    common_group: str,
    output_path: str,
    pixel_size_um: float = 60,
    vmax_percentile: float = 99.9,
    normalize_tic: bool = False,
    normalization_mode: str = 'individual',
    cmap: Any = None
) -> None:
    """Create figure using GridSpec to avoid tight_layout issues with colorbars."""
    if cmap is None:
        cmap = CUSTOM_CMAP
    
    mz_val = float(common_group)
    n_samples = len(pivot_df.index)
    cols = 4
    rows = int(np.ceil(n_samples / cols))
    
    # Pre-extract all data
    all_data = {}
    for sample_id in pivot_df.index:
        adata = sample_map.get(sample_id)
        if adata is None:
            continue
        val = pivot_df.loc[sample_id, mz_val]
        if pd.isna(val):
            continue
        x, y, intensities = extract_spatial_mz_data(adata, val, tol=0.01, normalize_tic=normalize_tic)
        if x is not None:
            all_data[sample_id] = (x, y, intensities)
    
    if not all_data:
        print(f"No data found for {common_group}, skipping...")
        return
    
    # Calculate vmax based on normalization mode
    if normalization_mode == 'global':
        all_intensities = np.concatenate([data[2] for data in all_data.values()])
        global_vmax = np.percentile(all_intensities, vmax_percentile)
        vmax_dict = {sid: global_vmax for sid in all_data.keys()}
    elif normalization_mode == 'row':
        vmax_dict = {}
        for row_idx in range(rows):
            start_idx = row_idx * cols
            end_idx = min(start_idx + cols, n_samples)
            row_sample_ids = [pivot_df.index[i] for i in range(start_idx, end_idx)]
            row_data = [all_data[sid][2] for sid in row_sample_ids if sid in all_data]
            if row_data:
                row_vmax = np.percentile(np.concatenate(row_data), vmax_percentile)
                for sid in row_sample_ids:
                    if sid in all_data:
                        vmax_dict[sid] = row_vmax
    else:  # individual
        vmax_dict = {sid: np.percentile(data[2], vmax_percentile) for sid, data in all_data.items()}
    
    # Create figure with GridSpec
    if normalization_mode == 'global':
        fig = plt.figure(figsize=(cols * 5 + 1, rows * 4))
        gs = GridSpec(rows, cols + 1, figure=fig, width_ratios=[1]*cols + [0.05], 
                      wspace=0.15, hspace=0.25)
    else:
        fig = plt.figure(figsize=(cols * 5.5, rows * 4))
        gs = GridSpec(rows, cols, figure=fig, wspace=0.3, hspace=0.25)
    
    scatter_collections = []
    axes_list = []
    
    for i, sample_id in enumerate(pivot_df.index):
        row_idx = i // cols
        col_idx = i % cols
        ax = fig.add_subplot(gs[row_idx, col_idx])
        axes_list.append(ax)
        
        if sample_id not in all_data:
            ax.axis('off')
            scatter_collections.append(None)
            continue
        
        x, y, intensities = all_data[sample_id]
        val = pivot_df.loc[sample_id, mz_val]
        vmax = vmax_dict.get(sample_id, np.percentile(intensities, vmax_percentile))
        
        spacing = get_data_spacing(x, y, pixel_size_um)
        
        # Set axis limits with half-pixel padding for tight arrangement
        padding = spacing * 0.5
        ax.set_xlim(x.min() - padding, x.max() + padding)
        ax.set_ylim(y.min() - padding, y.max() + padding)
        ax.set_aspect('equal')
        
        sc = ax.scatter(
            x, y,
            c=intensities,
            cmap=cmap,
            marker='s',
            s=1,  # Will be updated
            edgecolor='none',
            vmin=0,
            vmax=vmax
        )
        scatter_collections.append(sc)
        
        ax.set_title(f"{sample_id} m/z={val:.4f}", fontsize=10)
        ax.axis('off')
    
    # Add title
    title_suffix = " TIC Normalized" if normalize_tic else ""
    mode_suffix = f" ({normalization_mode.capitalize()})" if normalization_mode != 'individual' else ""
    fig.suptitle(f"Common Group: {common_group}{title_suffix}{mode_suffix}", fontsize=14, y=0.98)
    
    # Draw canvas to get accurate transforms
    fig.canvas.draw()
    
    # Update marker sizes for tight pixels
    for i, (ax, sc) in enumerate(zip(axes_list, scatter_collections)):
        if sc is None:
            continue
        sample_id = pivot_df.index[i]
        if sample_id not in all_data:
            continue
        x, y, _ = all_data[sample_id]
        spacing = get_data_spacing(x, y, pixel_size_um)
        marker_size = calculate_marker_size(ax, spacing, fig)
        sc.set_sizes([marker_size])
    
    # Add colorbars
    if normalization_mode == 'global':
        valid_sc = next((sc for sc in scatter_collections if sc is not None), None)
        if valid_sc:
            cbar_ax = fig.add_subplot(gs[:, -1])
            fig.colorbar(valid_sc, cax=cbar_ax)
    elif normalization_mode == 'row':
        for row_idx in range(rows):
            start_idx = row_idx * cols
            end_idx = min(start_idx + cols, n_samples)
            last_idx = end_idx - 1
            if last_idx < len(scatter_collections) and scatter_collections[last_idx] is not None:
                fig.colorbar(scatter_collections[last_idx], ax=axes_list[last_idx], 
                           fraction=0.046, pad=0.04)
    else:  # individual
        for i, sc in enumerate(scatter_collections):
            if sc is not None:
                fig.colorbar(sc, ax=axes_list[i], fraction=0.046, pad=0.04)
    
    # Save
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    fig.savefig(output_path, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.close(fig)


def process_all_visualizations(
    pivot_df: pd.DataFrame,
    sample_map: Dict[str, sc.AnnData],
    output_base_folder: str,
    pixel_size_um: float = 60,
    vmax_percentile: float = 99.9
) -> None:
    """Process all visualization modes for all m/z groups."""
    configs = [
        ('images_individual', False, 'individual'),
        ('images_individual_tic', True, 'individual'),
        ('images_row_tic', True, 'row'),
        ('images_all_tic', True, 'global'),
    ]
    
    total_mz = len(pivot_df.columns)
    
    for subfolder, normalize_tic, norm_mode in configs:
        output_folder = os.path.join(output_base_folder, subfolder)
        os.makedirs(output_folder, exist_ok=True)
        print(f"\nProcessing: {subfolder} ({total_mz} m/z groups)")
        
        for idx, common_group in enumerate(pivot_df.columns):
            if (idx + 1) % 50 == 0:
                print(f"  Progress: {idx + 1}/{total_mz}")
            
            output_path = os.path.join(output_folder, f"{common_group}.png")
            
            create_figure_with_gridspec(
                pivot_df=pivot_df,
                sample_map=sample_map,
                common_group=str(common_group),
                output_path=output_path,
                pixel_size_um=pixel_size_um,
                vmax_percentile=vmax_percentile,
                normalize_tic=normalize_tic,
                normalization_mode=norm_mode
            )
        
        print(f"  Completed: {subfolder}")


# =============================================================================
# MAIN FUNCTION
# =============================================================================

def run_visualization(
    input_folder: str = INPUT_FOLDER,
    common_mzs_csv_path: str = COMMON_MZS_CSV_PATH,
    results_output_folder: str = RESULTS_OUTPUT_FOLDER,
    pixel_size_um: float = PIXEL_SIZE_UM,
    vmax_percentile: float = VMAX_PERCENTILE
) -> None:
    """
    Main function to run the visualization pipeline.
    
    Parameters:
    -----------
    input_folder : str
        Path to folder containing h5ad files
    common_mzs_csv_path : str
        Path to CSV file with common m/z clusters
    results_output_folder : str
        Base folder for output images
    pixel_size_um : float
        Pixel size in micrometers (default: 60)
    vmax_percentile : float
        Percentile for vmax clipping (default: 99.9)
    """
    print("=" * 60)
    print("MSI Spatial Visualization")
    print(f"Pixel size: {pixel_size_um} µm")
    print(f"Vmax percentile: {vmax_percentile}")
    print("=" * 60)
    
    # Load samples
    print("\nLoading samples...")
    sample_ids = [
        "aad_1", "aad_2", "aad_3", "aad_4",
        "ac_1", "ac_2", "ac_3", "ac_4",
        "yad_1", "yad_2", "yad_3", "yad_4",
        "yc_1", "yc_2", "yc_3", "yc_4"
    ]
    sample_map = load_samples(input_folder, sample_ids)
    
    # Compute TIC
    print("\nComputing TIC values...")
    compute_tic(sample_map)
    
    # Load pivot table
    print("\nLoading common m/z data...")
    pivot_df = load_and_prepare_pivot_table(common_mzs_csv_path, SAMPLE_ORDER)
    print(f"Found {len(pivot_df.columns)} common m/z groups")
    
    # Process visualizations
    print("\nGenerating visualizations...")
    process_all_visualizations(
        pivot_df=pivot_df,
        sample_map=sample_map,
        output_base_folder=results_output_folder,
        pixel_size_um=pixel_size_um,
        vmax_percentile=vmax_percentile
    )
    
    print("\n" + "=" * 60)
    print("Processing complete!")
    print("=" * 60)


# =============================================================================
# EXECUTION
# =============================================================================

# For Jupyter notebook usage, just call run_visualization() directly:
# 
# run_visualization()
# 
# Or with custom parameters:
# 
# run_visualization(
#     pixel_size_um=60,
#     vmax_percentile=97.0
# )

# For command line usage only (won't run in Jupyter)
if __name__ == "__main__":
    import sys
    
    # Check if running in Jupyter/IPython
    def is_notebook():
        try:
            shell = get_ipython().__class__.__name__
            if shell == 'ZMQInteractiveShell':
                return True   # Jupyter notebook or qtconsole
            elif shell == 'TerminalInteractiveShell':
                return False  # Terminal running IPython
            else:
                return False
        except NameError:
            return False      # Standard Python interpreter
    
    if is_notebook():
        print("Running in Jupyter. Call run_visualization() to start.")
    else:
        # Command line execution with argparse
        import argparse
        
        parser = argparse.ArgumentParser(description="MSI Spatial Visualization")
        parser.add_argument('--pixel-size', type=float, default=PIXEL_SIZE_UM,
                           help=f'Pixel size in µm (default: {PIXEL_SIZE_UM})')
        parser.add_argument('--vmax-percentile', type=float, default=VMAX_PERCENTILE,
                           help=f'Percentile for vmax (default: {VMAX_PERCENTILE})')
        parser.add_argument('--input-folder', type=str, default=INPUT_FOLDER)
        parser.add_argument('--mz-csv', type=str, default=COMMON_MZS_CSV_PATH)
        parser.add_argument('--output-folder', type=str, default=RESULTS_OUTPUT_FOLDER)
        
        args = parser.parse_args()
        
        run_visualization(
            input_folder=args.input_folder,
            common_mzs_csv_path=args.mz_csv,
            results_output_folder=args.output_folder,
            pixel_size_um=args.pixel_size,
            vmax_percentile=args.vmax_percentile
        )

Running in Jupyter. Call run_visualization() to start.


In [3]:
run_visualization()

MSI Spatial Visualization
Pixel size: 60 µm
Vmax percentile: 99.9

Loading samples...
Loaded: aad_1
Loaded: aad_2
Loaded: aad_3
Loaded: aad_4
Loaded: ac_1
Loaded: ac_2
Loaded: ac_3
Loaded: ac_4
Loaded: yad_1
Loaded: yad_2
Loaded: yad_3
Loaded: yad_4
Loaded: yc_1
Loaded: yc_2
Loaded: yc_3
Loaded: yc_4

Computing TIC values...

Loading common m/z data...
Found 528 common m/z groups

Generating visualizations...

Processing: images_individual (528 m/z groups)
  Progress: 50/528
  Progress: 100/528
  Progress: 150/528
  Progress: 200/528
  Progress: 250/528
  Progress: 300/528
  Progress: 350/528
  Progress: 400/528
  Progress: 450/528
  Progress: 500/528
  Completed: images_individual

Processing: images_individual_tic (528 m/z groups)
  Progress: 50/528
  Progress: 100/528
  Progress: 150/528
  Progress: 200/528
  Progress: 250/528
  Progress: 300/528
  Progress: 350/528
  Progress: 400/528
  Progress: 450/528
  Progress: 500/528
  Completed: images_individual_tic

Processing: images_row

In [3]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
import plotly.express as px

# === INPUT PATHS ===
input_folder = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
common_mzs_csv_input_path = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/common_mz_clusters_improved.csv"
image_output_folder = "/home/ajarrah/PhD_Thesis/chapter_2/results/filtered_new_half/vmax/images_individual"
results_output_folder = "/home/ajarrah/PhD_Thesis/chapter_2/results/filtered_new_half/vmax/"

In [4]:
# Your list of sample IDs (also used as keys in the dictionary)
aad_1 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_aad_1_filtered_common.h5ad"))
aad_2 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_aad_2_filtered_common.h5ad"))
aad_3 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_aad_3_filtered_common.h5ad"))
aad_4 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_aad_4_filtered_common.h5ad"))
ac_1 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_ac_1_filtered_common.h5ad"))
ac_2 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_ac_2_filtered_common.h5ad"))
ac_3 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_ac_3_filtered_common.h5ad"))
ac_4 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_ac_4_filtered_common.h5ad"))
yad_1 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_yad_1_filtered_common.h5ad"))
yad_2 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_yad_2_filtered_common.h5ad"))
yad_3 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_yad_3_filtered_common.h5ad"))
yad_4 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_yad_4_filtered_common.h5ad"))
yc_1 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_yc_1_filtered_common.h5ad"))
yc_2 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_yc_2_filtered_common.h5ad"))
yc_3 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_yc_3_filtered_common.h5ad"))
yc_4 = sc.read_h5ad(os.path.join(input_folder, "halfbrain_yc_4_filtered_common.h5ad"))

In [6]:
sample_list = [
    aad_1, aad_2, aad_3, aad_4,
    ac_1, ac_2, ac_3, ac_4,
    yad_1, yad_2, yad_3, yad_4,
    yc_1, yc_2, yc_3, yc_4
]

sample_ids = [
    "aad_1", "aad_2", "aad_3", "aad_4",
    "ac_1", "ac_2", "ac_3", "ac_4",
    "yad_1", "yad_2", "yad_3", "yad_4",
    "yc_1", "yc_2", "yc_3", "yc_4"
]

# Map sample ID → AnnData object
sample_map = dict(zip(sample_ids, sample_list))


In [8]:
ac_1.obs

,x,y,TIC,sample,batch,age_group,disease_status,leiden_res_0.02,half_brain,x_um,y_um
103,3,0,6.962714e+06,3-1,Slide_3,Aged,Control,1,1,180,0
171,0,1,6.356645e+06,3-1,Slide_3,Aged,Control,1,1,0,60
172,1,1,6.302149e+06,3-1,Slide_3,Aged,Control,1,1,60,60
173,2,1,6.243896e+06,3-1,Slide_3,Aged,Control,1,1,120,60
174,3,1,6.657543e+06,3-1,Slide_3,Aged,Control,1,1,180,60
...,...,...,...,...,...,...,...,...,...,...,...
15867,18,95,4.090246e+06,3-1,Slide_3,Aged,Control,1,1,1080,5700
15868,19,95,4.156123e+06,3-1,Slide_3,Aged,Control,1,1,1140,5700
15869,20,95,4.657393e+06,3-1,Slide_3,Aged,Control,1,1,1200,5700
15870,21,95,4.214100e+06,3-1,Slide_3,Aged,Control,1,1,1260,5700


In [ ]:
# Assuming sample_list contains your AnnData objects and their variable names are like 'aad_1', 'ac_1', etc.
# They must be defined variables or in a dictionary for easy iteration.

# Example: if you have them in a dictionary like sample_map = {'aad_1': aad_1, ...}

for sample_id in sample_list:
    adata = sample_id  # Get the AnnData object by variable name string
    # Sum all intensities for each spot (row) across all m/z (columns)
    tic = adata.X.sum(axis=1)  # shape (n_spots, 1) or (n_spots,) depending on sparse or dense
    
    # If sparse, convert to flat array
    if hasattr(tic, "A1"):  # sparse matrix
        tic = tic.A1
    else:
        tic = np.asarray(tic).flatten()

    # Add TIC as a new column in obs
    adata.obs["Processed_TIC"] = tic

In [ ]:
# === STEP 3: Load common m/z list ===
common_mz_df = pd.read_csv(common_mzs_csv_input_path)
common_mz_df

In [ ]:

# Example: convert group/sample to sample_id that matches your sample_map keys
def make_sample_id(row):
    # lower group, then underscore, then sample number
    return f"{row['group'].lower()}_{row['sample']}"

common_mz_df['sample_id'] = common_mz_df.apply(make_sample_id, axis=1)
print(common_mz_df)

In [ ]:
pivot_df = common_mz_df.pivot(index='sample_id', columns='common_group_name', values='mzs')
pivot_df

In [ ]:
order = [
    'yc_1', 'yc_2', 'yc_3', 'yc_4',
    'yad_1', 'yad_2', 'yad_3', 'yad_4',
    'ac_1', 'ac_2', 'ac_3', 'ac_4',
    'aad_1', 'aad_2', 'aad_3', 'aad_4'
]

# Assuming pivot_df is your pivoted DataFrame with index = sample_id
pivot_df_sorted = pivot_df.loc[pivot_df.index.intersection(order)]
pivot_df_sorted = pivot_df_sorted.loc[order]

pivot_df_sorted

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
# === COLOR SCALE ===
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),   # white
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),  # navy
    (0.15, "#0000FF"),  # blue
    (0.30, "#8000FF"),  # purple-ish
    (0.45, "#FF0000"),  # red
    (0.60, "#FF8000"),  # orange
    (0.75, "#FFFF00"),  # yellow
    (1.0, "#FFFFFF")   # white
])

In [ ]:
# Create output folder if it doesn't exist
os.makedirs(image_output_folder, exist_ok=True)

In [ ]:
pivot_df_sorted.columns

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

def extract_spatial_mz_data(adata, mz_val, tol=0.01):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - mz_val)
    if np.min(mz_diff) > tol:
        return None, None, None
    mz_index = np.argmin(mz_diff)
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    return x, y, intensities

# Iterate over columns (common_group_name)
for common_group in pivot_df_sorted.columns:
    mz_val = float(common_group)  # convert column name to float if needed

    # Prepare subplots: 4x4 grid (adjust if you have a different number of samples)
    n_samples = len(pivot_df_sorted.index)
    cols = 4
    rows = int(np.ceil(n_samples / cols))

    fig, axs = plt.subplots(rows, cols, figsize=(cols*6, rows*4), squeeze=False)
    axs = axs.flatten()

    for i, sample_id in enumerate(pivot_df_sorted.index):
        adata = sample_map.get(sample_id)

        val = pivot_df_sorted.loc[sample_id, mz_val]
        x, y, intensities = extract_spatial_mz_data(adata, val, tol=0.001)

        # Compute vmax as the 97th percentile (top 3% saturated)
        vmax_value = np.percentile(intensities, 99.9)

        ax = axs[i]
        sc = ax.scatter(x, y, c=intensities, cmap=custom_cmap, marker='s', s=8, edgecolor='none',
                        vmin=0, vmax=vmax_value)
        ax.set_title(f"{sample_id} m/z={val:.4f}")
        ax.axis('off')

        plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)

    # Turn off any unused subplots
    for j in range(i+1, len(axs)):
        axs[j].axis('off')

    fig.suptitle(f"Common Group: {common_group}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # Save figure
    fig.savefig(os.path.join(image_output_folder, f"{common_group}.png"))
    plt.close(fig)


In [ ]:
os.makedirs(image_output_folder + '_tic', exist_ok=True)

def extract_spatial_mz_data_tic(adata, mz_val, tol=0.01):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - mz_val)
    if np.min(mz_diff) > tol:
        return None, None, None
    mz_index = np.argmin(mz_diff)
    
    # Extract raw intensities for the mz_val
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    
    # Extract spatial coordinates
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    
    # Extract TIC for each spot (assuming column name is 'Processed_TIC')
    tic = adata.obs["Processed_TIC"].values
    
    # Normalize intensities by TIC, avoid division by zero
    normalized_intensities = intensities / tic
    normalized_intensities = np.nan_to_num(normalized_intensities, nan=0.0, posinf=0.0, neginf=0.0)
    
    return x, y, normalized_intensities

# Iterate over columns (common_group_name)
for common_group in pivot_df_sorted.columns:
    mz_val = float(common_group)  # convert column name to float if needed

    # Prepare subplots: 4x4 grid (adjust if you have a different number of samples)
    n_samples = len(pivot_df_sorted.index)
    cols = 4
    rows = int(np.ceil(n_samples / cols))

    fig, axs = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4), squeeze=False)
    axs = axs.flatten()

    for i, sample_id in enumerate(pivot_df_sorted.index):
        adata = sample_map.get(sample_id)
        
        val = pivot_df_sorted.loc[sample_id, mz_val]
        x, y, intensities = extract_spatial_mz_data_tic(adata, val, tol=0.001)

        # Compute vmax as 97th percentile (top 3% saturates)
        vmax_value = np.percentile(intensities, 99.9)

        ax = axs[i]
        sc = ax.scatter(x, y, c=intensities, cmap=custom_cmap, marker='s', s=8, edgecolor='none',
                        vmin=0, vmax=vmax_value)
        ax.set_title(f"{sample_id} m/z={val:.4f}")
        ax.axis('off')

        plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)

    # Turn off any unused subplots
    for j in range(i + 1, len(axs)):
        axs[j].axis('off')

    fig.suptitle(f"Common Group: {common_group} TIC Normalized", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # Save figure
    fig.savefig(os.path.join(image_output_folder + '_tic/', f"{common_group}.png"))
    plt.close(fig)


In [ ]:
os.makedirs(results_output_folder + 'images_row_tic', exist_ok=True)

def extract_spatial_mz_data_tic(adata, mz_val, tol=0.01):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - mz_val)
    if np.min(mz_diff) > tol:
        return None, None, None
    mz_index = np.argmin(mz_diff)
    
    # Extract raw intensities for the mz_val
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    
    # Extract spatial coordinates
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    
    # Extract TIC for each spot (assuming column name is 'Processed_TIC')
    tic = adata.obs["Processed_TIC"].values
    
    # Normalize intensities by TIC, avoid division by zero
    normalized_intensities = intensities / tic
    normalized_intensities = np.nan_to_num(normalized_intensities, nan=0.0, posinf=0.0, neginf=0.0)
    
    return x, y, normalized_intensities

for common_group in pivot_df_sorted.columns:
    mz_val = float(common_group)

    n_samples = len(pivot_df_sorted.index)
    cols = 4
    rows = int(np.ceil(n_samples / cols))

    fig, axs = plt.subplots(rows, cols, figsize=(cols * 6.5, rows * 4), squeeze=False)
    axs = axs.flatten()

    for row_idx in range(rows):
        start_idx = row_idx * cols
        end_idx = min(start_idx + cols, n_samples)
        row_sample_ids = pivot_df_sorted.index[start_idx:end_idx]

        # --- Compute vmax for this row using 97th percentile ---
        all_intensities = []
        for sample_id in row_sample_ids:
            adata = sample_map.get(sample_id)
            val = pivot_df_sorted.loc[sample_id, mz_val]
            _, _, intensities = extract_spatial_mz_data_tic(adata, val, tol=0.001)
            all_intensities.extend(intensities)
        vmax = np.percentile(all_intensities, 99.9)  # top 3% clipped
        vmin = 0

        # --- Plot each sample in this row ---
        for col_idx, sample_id in enumerate(row_sample_ids):
            i = start_idx + col_idx
            adata = sample_map.get(sample_id)
            val = pivot_df_sorted.loc[sample_id, mz_val]
            x, y, intensities = extract_spatial_mz_data_tic(adata, val, tol=0.001)

            ax = axs[i]
            sc = ax.scatter(x, y, c=intensities, cmap=custom_cmap, marker='s', s=8, edgecolor='none',
                            vmin=vmin, vmax=vmax)
            ax.set_title(f"{sample_id} m/z={val:.4f}")
            ax.axis('off')

        # Add colorbar for the row (last axis in row)
        last_ax_in_row = axs[start_idx + len(row_sample_ids) - 1]
        fig.colorbar(sc, ax=last_ax_in_row, fraction=0.046, pad=0.04)

    # Turn off unused subplots
    for j in range(n_samples, rows * cols):
        axs[j].axis('off')

    fig.suptitle(f"Common Group: {common_group} TIC Normalized", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    fig.savefig(os.path.join(results_output_folder + 'images_row_tic/', f"{common_group}.png"))
    plt.close(fig)

In [ ]:
os.makedirs(results_output_folder + 'images_all_tic', exist_ok=True)

def extract_spatial_mz_data_tic(adata, mz_val, tol=0.01):
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - mz_val)
    if np.min(mz_diff) > tol:
        return None, None, None
    mz_index = np.argmin(mz_diff)
    
    # Extract raw intensities for the mz_val
    intensities = adata.X[:, mz_index].toarray().flatten() if hasattr(adata.X, "toarray") else adata.X[:, mz_index]
    
    # Extract spatial coordinates
    x = adata.obs["x"].values
    y = adata.obs["y"].values
    
    # Extract TIC for each spot (assuming column name is 'TIC')
    tic = adata.obs["Processed_TIC"].values
    
    # Normalize intensities by TIC, avoid division by zero
    normalized_intensities = intensities / tic
    normalized_intensities = np.nan_to_num(normalized_intensities, nan=0.0, posinf=0.0, neginf=0.0)
    
    return x, y, normalized_intensities

for common_group in pivot_df_sorted.columns:
    mz_val = float(common_group)

    n_samples = len(pivot_df_sorted.index)
    cols = 4
    rows = int(np.ceil(n_samples / cols))

    fig, axs = plt.subplots(rows, cols, figsize=(cols * 6.5, rows * 4), squeeze=False)
    axs = axs.flatten()

    # --- Compute global vmax using 97th percentile (clipping top 3%) ---
    all_intensities = []
    for sample_id in pivot_df_sorted.index:
        adata = sample_map.get(sample_id)
        val = pivot_df_sorted.loc[sample_id, mz_val]
        _, _, intensities = extract_spatial_mz_data_tic(adata, val, tol=0.001)
        all_intensities.extend(intensities)

    vmax = np.percentile(all_intensities, 99.9)  # 97th percentile
    vmin = 0

    # --- Plot all samples ---
    for i, sample_id in enumerate(pivot_df_sorted.index):
        adata = sample_map.get(sample_id)
        val = pivot_df_sorted.loc[sample_id, mz_val]
        x, y, intensities = extract_spatial_mz_data_tic(adata, val, tol=0.001)

        ax = axs[i]
        sc = ax.scatter(x, y, c=intensities, cmap=custom_cmap, marker='s', s=8, edgecolor='none',
                        vmin=vmin, vmax=vmax)
        ax.set_title(f"{sample_id} m/z={val:.4f}")
        ax.axis('off')

    # Turn off unused subplots
    for j in range(n_samples, rows * cols):
        axs[j].axis('off')

    # Place ONE colorbar outside the grid at far right
    cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])  # [left, bottom, width, height]
    fig.colorbar(sc, cax=cbar_ax)

    fig.suptitle(f"Common Group: {common_group} TIC Normalized", fontsize=16)
    plt.tight_layout(rect=[0, 0, 0.9, 0.96])  # leave space on right for cbar

    fig.savefig(os.path.join(results_output_folder + 'images_all_tic/', f"{common_group}.png"))
    plt.close(fig)
